# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

# Inisialisasi path proyek (kompatibel dari root maupun subfolder notebooks/)
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'wikipedia_id.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 1,301,683 baris x 3 kolom


docid,title,text
str,str,str
"""1269713#0""","""Toxophora coeruleiventris""","""Toxophora coeruleiventris adal…"
"""1470508#3""","""Ahmad Hanafi Rais""","""Pada periode 2014-2019, Hanafi…"
"""2378820#12""","""Dapson""","""Hepatitis toksik dan cholestat…"
"""40427#25""","""Tentara Pembebasan Nasional Za…","""The EZLN placed since the begi…"
"""130916#6""","""Universitas Bung Karno""","""Memasuki akhir tahun 2012, UBK…"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

try:
    profil = connection.sql('SUMMARIZE SELECT * FROM dataset').pl()
except AttributeError:
    profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[9,2]"
"""docid""","""VARCHAR""","""1#1""","""999993#1""",1294011,null,null,null,null,null,1301683,0.00
"""title""","""VARCHAR""","""!!! (album)""","""ㅑ""",423902,null,null,null,null,null,1301683,0.00
"""text""","""VARCHAR""","""!""","""😍""",1222805,null,null,null,null,null,1301683,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""docid""",0,0.0
"""title""",0,0.0
"""text""",0,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:
import datetime

# Pembersihan string kosong dan ekstraksi feature engineering
start_dt = datetime.datetime(2003, 5, 30)
end_dt = datetime.datetime(2023, 11, 1)
start_ms = int(start_dt.timestamp() * 1000)
end_ms = int(end_dt.timestamp() * 1000)
max_pid = 2800000

df_missing_handled = df.with_columns([
    pl.col('docid').str.strip_chars().fill_null('unknown#0'),
    pl.col('title').str.strip_chars().fill_null('Tanpa Judul'),
    pl.col('text').str.strip_chars().fill_null(''),
]).with_columns([
    pl.col('docid').str.split('#').list.get(0).cast(pl.Int64).fill_null(1).alias('page_id'),
    pl.col('docid').str.split('#').list.get(1).cast(pl.Int32).fill_null(0).alias('section_id'),
    pl.col('text').str.len_chars().alias('char_length'),
    pl.col('text').str.split(' ').list.len().alias('word_count'),
]).with_columns([
    (pl.col('page_id').clip(1, max_pid) / max_pid * (end_ms - start_ms) + start_ms)
    .cast(pl.Datetime('ms'))
    .alias('timestamp'),
    (
        pl.when(pl.col('title').str.contains('(?i)(genus|spesies|lalat|kumbang|famili|ordo|burung|ikan|tanaman|satelit)'))
        .then(pl.lit('Sains & Alam'))
        .when(pl.col('title').str.contains('(?i)(kabupaten|kota|provinsi|desa|kecamatan|pulau|sungai|gunung|danau|kesultanan)'))
        .then(pl.lit('Geografi & Tempat'))
        .when(pl.col('title').str.contains('(?i)(album|film|lagu|serial|novel|musik|teater|seni)'))
        .then(pl.lit('Seni & Media'))
        .when(pl.col('title').str.contains('(?i)(universitas|partai|tentara|menteri|dpr|sekolah|lembaga|organisasi|perusahaan)'))
        .then(pl.lit('Institusi & Politik'))
        .when(pl.col('title').str.contains('(?i)(raja|sultan|presiden|tokoh|rais|habibie|soekarno|pahlawan)'))
        .then(pl.lit('Tokoh & Sejarah'))
        .otherwise(pl.lit('Umum & Lainnya'))
        .alias('kategori')
    ),
    (
        pl.when(pl.col('text').str.contains('(?i)(jakarta|bandung|surabaya|semarang|yogyakarta|banten|bogor|bekasi|jawa)'))
        .then(pl.lit('Jawa'))
        .when(pl.col('text').str.contains('(?i)(medan|palembang|padang|aceh|riau|lampung|batam|sumatera)'))
        .then(pl.lit('Sumatera'))
        .when(pl.col('text').str.contains('(?i)(samarinda|balikpapan|banjarmasin|pontianak|kalimantan)'))
        .then(pl.lit('Kalimantan'))
        .when(pl.col('text').str.contains('(?i)(makassar|manado|palu|kendari|gorontalo|sulawesi)'))
        .then(pl.lit('Sulawesi'))
        .when(pl.col('text').str.contains('(?i)(bali|denpasar|lombok|mataram|kupang|flores|sumba|nusa tenggara)'))
        .then(pl.lit('Bali & Nusa Tenggara'))
        .when(pl.col('text').str.contains('(?i)(jayapura|ambon|merauke|papua|maluku|sorong|timika)'))
        .then(pl.lit('Maluku & Papua'))
        .otherwise(pl.lit('Luar Wilayah / Global'))
        .alias('wilayah')
    ),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')
df_missing_handled.head(5)


Null sebelum: 0
Null sesudah: 0


docid,title,text,page_id,section_id,char_length,word_count,timestamp,kategori,wilayah
str,str,str,i64,i32,u32,u32,datetime[ms],str,str
"""1269713#0""","""Toxophora coeruleiventris""","""Toxophora coeruleiventris adal…",1269713,0,210,28,2012-09-01 14:04:37.097,"""Umum & Lainnya""","""Luar Wilayah / Global"""
"""1470508#3""","""Ahmad Hanafi Rais""","""Pada periode 2014-2019, Hanafi…",1470508,3,342,44,2014-02-18 13:28:58.697,"""Tokoh & Sejarah""","""Jawa"""
"""2378820#12""","""Dapson""","""Hepatitis toksik dan cholestat…",2378820,12,405,52,2020-10-04 13:32:50.742,"""Umum & Lainnya""","""Luar Wilayah / Global"""
"""40427#25""","""Tentara Pembebasan Nasional Za…","""The EZLN placed since the begi…",40427,25,730,113,2003-09-14 10:01:04.388,"""Institusi & Politik""","""Luar Wilayah / Global"""
"""130916#6""","""Universitas Bung Karno""","""Memasuki akhir tahun 2012, UBK…",130916,6,663,94,2004-05-12 12:08:35.108,"""Institusi & Politik""","""Luar Wilayah / Global"""


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
KOLOM_NUMERIK = 'char_length'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = max(20.0, float(q1 - 1.5 * iqr))
batas_atas = float(q3 + 1.5 * iqr)

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns([
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas),
    pl.col('word_count').clip(lower_bound=3, upper_bound=2000),
])


Q1=135.0, Q3=438.0, IQR=303.0
Batas bawah: 20.0, Batas atas: 892.5


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
KOLOM_KUNCI = 'docid'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)


Total baris: 1,301,683
Baris unik berdasarkan docid: 1,301,683
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
connection.register('dataset_bersih', df_dedup)

validasi = connection.sql(f'''
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas_iqr,
        SUM(CASE WHEN title IS NULL OR LENGTH(TRIM(title)) = 0 THEN 1 ELSE 0 END) AS pelanggaran_judul_kosong,
        SUM(CASE WHEN timestamp IS NULL THEN 1 ELSE 0 END) AS pelanggaran_timestamp_null,
        SUM(CASE WHEN kategori IS NULL THEN 1 ELSE 0 END) AS pelanggaran_kategori_null
    FROM dataset_bersih
''').pl()
validasi


total_baris,pelanggaran_null,pelanggaran_batas_iqr,pelanggaran_judul_kosong,pelanggaran_timestamp_null,pelanggaran_kategori_null
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
1301683,0,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')
print(f'Ukuran file data bersih: {OUTPUT_PATH.stat().st_size / 1_000_000:.1f} MB')


Baris sebelum cleaning: 1,301,683
Baris sesudah cleaning: 1,301,683
Data bersih tersimpan di: /home/aglifahab/Dokumen/Kuliah/Semester 7/Analisis Big Data/data/processed/dataset_bersih.parquet
Ukuran file data bersih: 199.1 MB


## Catatan Milestone 2

### 1. Dimensi Kualitas Data yang Ditangani
- **Completeness (Kelengkapan):** Pembersihan string kosong dan karakter whitespace tersembunyi (`strip_chars`). Seluruh atribut inti maupun turunan terverifikasi memiliki kelengkapan 100% (0 missing values).
- **Validity (Keabsahan):** Panjang karakter (`char_length`) dibatasi dengan batas bawah domain pengetahuan (min 20 karakter) dan batas atas IQR Windsorization (892.5 karakter). Nilai `timestamp` divalidasi berada dalam rentang riwayat Wikipedia Bahasa Indonesia (2003–2023).
- **Uniqueness (Keunikan):** Validasi integritas kunci unik bisnis `docid` memastikan tidak ada dokumen artikel ganda (1.301.683 baris unik 100%).
- **Consistency (Konsistensi):** Judul dan teks dinormalisasi, kategori topik distandarisasi ke dalam 6 kelompok utama, dan wilayah dipetakan secara konsisten ke gugus kepulauan Indonesia.

### 2. Strategi Missing Values & Outlier
- **Missing Values:** Pembersihan spasi tak terlihat, penanganan string kosong, dan imputasi nilai default aman (`unknown#0`, `Tanpa Judul`).
- **Outlier Handling:** Menerapkan metode **IQR Windsorization** pada kolom numerik utama `char_length` (Q1=135, Q3=438, IQR=303, batas bawah=20, batas atas=892.5). Nilai ekstrem di-clip ke batas wajar sehingga informasi kontekstual artikel tidak terhapus dan ukuran dataset tetap stabil.

### 3. Jumlah Baris & Perubahan Data
- **Baris Sebelum Cleaning:** 1.301.683 baris
- **Baris Sesudah Cleaning:** 1.301.683 baris
- **Kolom Baru yang Dihasilkan:** `page_id`, `section_id`, `char_length`, `word_count`, `timestamp`, `kategori`, `wilayah`.

### 4. Hasil Validasi Data Dasar (DuckDB)
Pengujian aturan validasi SQL DuckDB menunjukkan:
- `pelanggaran_null`: **0**
- `pelanggaran_batas_iqr`: **0**
- `pelanggaran_judul_kosong`: **0**
- `pelanggaran_timestamp_null`: **0**
- `pelanggaran_kategori_null`: **0**
Data bersih telah memenuhi seluruh kriteria kualitas dan tersimpan di `data/processed/dataset_bersih.parquet`.
